In [1]:
import warnings
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Install required packages for Google Document AI
# We'll use the Terminal tool to install google-cloud-documentai and pandas for later use.

from metagpt.tools.libs.terminal import Terminal

terminal = Terminal()
await terminal.run_command("pip install google-cloud-documentai pandas")

2025-12-26 12:09:07.688 | INFO     | metagpt.const:get_metagpt_package_root:15 - Package root set to /home/metagpt-fuzzing/Desktop/MetaGPT
2025-12-26 12:09:07.691 | INFO     | metagpt.const:get_metagpt_package_root:15 - Package root set to /home/metagpt-fuzzing/Desktop/MetaGPT
2025-12-26 12:09:09.259 | INFO     | metagpt.tools.libs.terminal:_check_state:55 - The terminal is at:


'Collecting google-cloud-documentai\n  Downloading google_cloud_documentai-3.7.0-py3-none-any.whl.metadata (9.8 kB)\nRequirement already satisfied: pandas in /home/metagpt-fuzzing/anaconda3/envs/metagpt/lib/python3.11/site-packages (2.1.1)\nRequirement already satisfied: google-api-core!=2.0.*,!=2.1.*,!=2.10.*,!=2.2.*,!=2.3.*,!=2.4.*,!=2.5.*,!=2.6.*,!=2.7.*,!=2.8.*,!=2.9.*,<3.0.0,>=1.34.1 in /home/metagpt-fuzzing/anaconda3/envs/metagpt/lib/python3.11/site-packages (from google-api-core[grpc]!=2.0.*,!=2.1.*,!=2.10.*,!=2.2.*,!=2.3.*,!=2.4.*,!=2.5.*,!=2.6.*,!=2.7.*,!=2.8.*,!=2.9.*,<3.0.0,>=1.34.1->google-cloud-documentai) (2.28.1)\nRequirement already satisfied: google-auth!=2.24.0,!=2.25.0,<3.0.0,>=2.14.1 in /home/metagpt-fuzzing/anaconda3/envs/metagpt/lib/python3.11/site-packages (from google-cloud-documentai) (2.45.0)\nRequirement already satisfied: grpcio<2.0.0,>=1.33.2 in /home/metagpt-fuzzing/anaconda3/envs/metagpt/lib/python3.11/site-packages (from google-cloud-documentai) (1.67.1)

In [3]:
import os
from google.cloud import documentai_v1beta3 as documentai

# Step 2: Authenticate with Google Cloud Document AI
# Set the environment variable for authentication (replace with your actual path)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path/to/your/service-account-key.json"

# Step 3: Define a function to process supplier invoice files (image or PDF)
def process_invoice_document(
    project_id: str,
    location: str,
    processor_id: str,
    file_path: str,
    mime_type: str = "application/pdf"
):
    """
    Processes a supplier invoice using Google Document AI and extracts specified fields.
    Args:
        project_id (str): Google Cloud project ID.
        location (str): Processor location (e.g., 'us').
        processor_id (str): Document AI processor ID.
        file_path (str): Path to the invoice file (PDF or image).
        mime_type (str): MIME type of the file.
    Returns:
        dict: Extracted fields.
    """
    client = documentai.DocumentUnderstandingServiceClient()
    name = f"projects/{project_id}/locations/{location}/processors/{processor_id}"

    # Read the file as bytes
    with open(file_path, "rb") as f:
        file_content = f.read()

    # Prepare the request
    raw_document = documentai.RawDocument(content=file_content, mime_type=mime_type)
    request = documentai.ProcessRequest(name=name, raw_document=raw_document)

    # Process the document
    result = client.process_document(request=request)
    document = result.document

    # Extract fields (robust to scan conditions)
    fields = {
        "invoice_number": None,
        "date": None,
        "supplier_name": None,
        "total_amount": None
    }
    for entity in document.entities:
        entity_type = entity.type_.lower()
        value = entity.mention_text
        if "invoice_number" in entity_type or "invoice no" in entity_type:
            fields["invoice_number"] = value
        elif "date" in entity_type:
            fields["date"] = value
        elif "supplier" in entity_type or "vendor" in entity_type:
            fields["supplier_name"] = value
        elif "total" in entity_type or "amount" in entity_type:
            fields["total_amount"] = value

    return fields

# Example usage (replace with your actual values and file path)
# result = process_invoice_document(
#     project_id="your-project-id",
#     location="us",
#     processor_id="your-processor-id",
#     file_path="path/to/invoice.pdf",
#     mime_type="application/pdf"
# )
# print(result)

In [4]:
import json
import pandas as pd

def save_invoice_data(fields: dict, json_path: str, csv_path: str):
    """
    Save extracted invoice fields to both JSON and CSV files.
    Args:
        fields (dict): Extracted invoice fields.
        json_path (str): Path to save JSON file.
        csv_path (str): Path to save CSV file.
    """
    # Save as JSON
    with open(json_path, "w") as jf:
        json.dump(fields, jf, indent=4)
    
    # Save as CSV (single row)
    df = pd.DataFrame([fields])
    df.to_csv(csv_path, index=False)

# Example usage:
# fields = process_invoice_document(project_id, location, processor_id, file_path, mime_type)
# save_invoice_data(fields, "invoice_result.json", "invoice_result.csv")